In [ ]:
!pip install --upgrade --force-reinstall -q streamlit langchain langchain-google-genai langchain-community pypdf faiss-cpu

In [ ]:
!pip install groq sentence-transformers

In [ ]:
!pip install wikipedia

In [ ]:
!pip install pymupdf

In [25]:
%%writefile app.py

import streamlit as st
import os
import wikipedia
import re
import requests

from groq import Groq

from langchain_community.document_loaders import (
    PyMuPDFLoader,  # Swapped for blazing-fast 500+ page parsing
    WebBaseLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Smart Multi-Source RAG Chatbot",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 Smart Multi-Source RAG Chatbot")

st.write("""
📄 High-Capacity PDF 🌐 Website 📚 Wikipedia \n
✔ Optimized Memory Management + Fast PyMuPDF Parsing + Stable + Production Ready
""")

# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:
    st.header("⚙️ Settings")

    groq_api_key = st.text_input(
        "Enter Groq API Key",
        type="password",
        placeholder="sk-xxxxxx"
    )

    st.subheader("🌐 Website URL")
    website_url = st.text_input(
        "Enter Website URL",
        placeholder="https://example.com"
    )

    st.subheader("📚 Wikipedia Topic")
    wikipedia_query = st.text_input(
        "Enter Wikipedia Topic",
        placeholder="Machine Learning, Pakistan, AI"
    )

# =========================================================
# SESSION STATE
# =========================================================

if "vector_store" not in st.session_state:
    st.session_state.vector_store = None

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# =========================================================
# FILE UPLOAD
# =========================================================

uploaded_file = st.file_uploader(
    "📄 Upload PDF File (Supports heavy files up to 500+ pages)",
    type=["pdf"]
)

# =========================================================
# EMBEDDINGS
# =========================================================

@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

# =========================================================
# URL VALIDATION
# =========================================================

def is_valid_url(url):
    return url and (url.startswith("http://") or url.startswith("https://"))

# =========================================================
# WIKIPEDIA LOADER (STABLE)
# =========================================================
def load_wikipedia(query):
    try:
        from urllib.parse import quote

        if not query:
            return []

        query = query.split("(")[0].strip()

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(query.replace(' ', '_'))}"

        r = requests.get(url, headers=headers, timeout=10)

        if r.status_code != 200:
            return []

        data = r.json()
        extract = data.get("extract")

        if not extract or len(extract) < 20:
            return []

        return [Document(
            page_content=extract,
            metadata={"source": "wikipedia"}
        )]

    except Exception:
        return []

# =========================================================
# BUILD KNOWLEDGE BASE
# =========================================================

def build_knowledge_base(pdf_file, url, wiki_query):
    docs = []

    # ---------------- PDF (Optimized for Huge Files) ----------------
    if pdf_file:
        try:
            temp_path = "temp_large_stream.pdf"

            # Stream chunk by chunk into local disk to minimize RAM consumption
            with open(temp_path, "wb") as f:
                bytes_data = pdf_file.read()
                f.write(bytes_data)

            # PyMuPDFLoader uses optimized C-libraries under the hood
            with st.spinner("Parsing PDF pages instantly using PyMuPDF..."):
                loader = PyMuPDFLoader(temp_path)
                docs.extend(loader.load())

            if os.path.exists(temp_path):
                os.remove(temp_path)

            st.success(f"✅ PDF Loaded Successfully ({len(docs)} pages parsed)")

        except Exception as e:
            st.error(f"❌ PDF Error: {e}")
            if os.path.exists(temp_path):
                os.remove(temp_path)

    # ---------------- WEBSITE ----------------
    if url:
        if not is_valid_url(url):
            st.warning("⚠️ Invalid URL! Use https://")
        else:
            try:
                loader = WebBaseLoader(url)
                docs.extend(loader.load())
                st.success("✅ Website Loaded Successfully")

            except Exception as e:
                st.error(f"❌ Website Error: {e}")

    # ---------------- WIKIPEDIA ----------------
    if wiki_query and len(wiki_query.strip()) > 2:
        with st.spinner("📚 Fetching Wikipedia data..."):
            wiki_docs = load_wikipedia(wiki_query)

        if wiki_docs:
            docs.extend(wiki_docs)
            st.success("✅ Wikipedia Loaded")
        else:
            st.warning("⚠️ getting no data from Wikipedia")

    return docs

# =========================================================
# PROCESS BUTTON
# =========================================================

if groq_api_key:

    if st.button("🚀 Process Knowledge Base"):

        with st.spinner("Processing documents into chunks..."):

            docs = build_knowledge_base(uploaded_file, website_url, wikipedia_query)

            if len(docs) == 0:
                st.warning("❌ No valid data found.")
            else:
                st.info(f"📊 Total raw pages/documents fetched: {len(docs)}")

                splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=200
                )

                chunks = splitter.split_documents(docs)
                st.info(f"✂️ Total sub-chunks generated: {len(chunks)}")

                embeddings = get_embeddings()

                # Batch ingestion to safeguard FAISS on high chunk loads
                with st.spinner("Building FAISS Vector Index (Processing batches)..."):
                    batch_size = 500

                    # Initialize the vector store with the first batch
                    vector_store = FAISS.from_documents(chunks[:batch_size], embeddings)

                    # Add remaining chunks iteratively in chunks of 500
                    for i in range(batch_size, len(chunks), batch_size):
                        batch = chunks[i:i + batch_size]
                        vector_store.add_documents(batch)

                    st.session_state.vector_store = vector_store

                st.success("🎉 Knowledge Base Ready! Even with 500+ pages, context tracking is smooth.")

# =========================================================
# CHAT FUNCTION (FIXED GROQ MODEL)
# =========================================================

def ask_llm(question, vector_store, api_key, history):

    client = Groq(api_key=api_key)

    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(question)

    context = "\n\n".join([d.page_content for d in docs])

    chat_history = "\n".join(
        [f"{h['role']}: {h['content']}" for h in history]
    )

    prompt = f"""
You are a helpful AI assistant.

Context:
{context}

Chat History:
{chat_history}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=1024
    )

    return response.choices[0].message.content

# =========================================================
# CHAT HISTORY
# =========================================================

for msg in st.session_state.chat_history:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# =========================================================
# CHAT INPUT
# =========================================================

user_input = st.chat_input("Ask something from your knowledge base...")

if user_input:

    if not groq_api_key:
        st.warning("⚠️ Enter Groq API Key")

    elif st.session_state.vector_store is None:
        st.warning("⚠️ Process knowledge base first")

    else:

        st.chat_message("user").write(user_input)

        st.session_state.chat_history.append({
            "role": "user",
            "content": user_input
        })

        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):

                answer = ask_llm(
                    user_input,
                    st.session_state.vector_store,
                    groq_api_key,
                    st.session_state.chat_history[:-1]
                )

                st.write(answer)

        st.session_state.chat_history.append({
            "role": "assistant",
            "content": answer
        })

Overwriting app.py


In [16]:
!sed -i 's/llama-3.1-70b-versatile/llama-3.3-70b-versatile/g' app.py

In [17]:
!pip install streamlit groq langchain langchain-community faiss-cpu sentence-transformers pypdf pyngrok -q

In [ ]:
!npm install -g localtunnel

In [19]:
!pkill streamlit
!fuser -k 8501/tcp
!fuser -k 8502/tcp
!fuser -k 8503/tcp

In [20]:
!pkill streamlit
!streamlit run app.py --server.port 8501 &>/dev/null &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3CrdOPyVVYlSXEDZakV6kjl2rfW_57xchSzzndTz6b5nEVxD7")

ngrok.kill()
public_url = ngrok.connect(8501)

print(public_url)